In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

In [2]:
base_model_id = "Qwen/Qwen3.5-27B"
adapter_id = None
lang = None

tokenizer = AutoTokenizer.from_pretrained(base_model_id)
base_model = AutoModelForCausalLM.from_pretrained(base_model_id, device_map="auto")

model = base_model
if adapter_id is not None:
    model = PeftModel.from_pretrained(base_model, adapter_id, subfolder=lang)

Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/851 [00:00<?, ?it/s]

In [9]:
#metamodel = MetaModel(os.path.join(TEXT2VQL_ROOT,'dataset_construction/seed/yakindu_simplified.ecore'))
instruction = """
Given the following Common Requirement Modeling Language mode, generate a CRML model of a air conditioning unit.

model Interaction_ECS_Consumer is {
    /* Abstract class for a system */
    partial class System is {
        /* Boolean that states whether the system is in operation.
        The values of the Boolean can be provided by a behavioral model of the system via the FMI, or by the system itself via a sensor */
        Boolean inOperation is external;
    };

    /* Class that defines the ECS for the purpose of the contract between the ECS and the consumer */
    class ECS is {
        /* The time dependent values for deltaT from the ECS.
        They are provided by a behavioral model of the ECS via the FMI, or by the ECS itself via sensors. */
        TemperatureDifference deltaT is external;
    } extends System;

    /* Class that defines the consumer for the purpose of the contract between the ECS and the consumer */
    class Consumer is {
        /* The time dependent values for deltaT from the consumer.
        They are provided by a behavioral model of the consumer via the FMI, or by the consumer itself via sensors. */
        TemperatureDifference deltaT is external;
    } extends System;

    /* Class that defines the contract between the ECS and the consumer */
    class Contract is {
        /* Instance of the ECS that cools the consumer */
        ECS ecs;
        /* Instance of the consumer cooled by the ECS */
        Consumer consumer;
        /* Conditions to be verified by the contract to ensure the proper functioning of the complete system ECS + consumer  */
        Boolean r1 is while consumer.inOperation ensure (ecs.inOperation and ecs.deltaT <= consumer.deltaT);
    };

    /* Create and instance of the ECS and an instance of the consumer */
    ECS ecs is new ECS;
    Consumer consumer is new Consumer;

    /* Create an instance of the contract between the ECS and the consumer.
    Once the contract is created, the conditions of the contract are active.
    The simulation must cover the time period while the consumer is in operation. */
    Contract contract is new Contract (ecs is ecs, consumer is consumer);
};
"""

inputs = tokenizer(instruction, return_tensors="pt").to(model.device)

outputs = model.generate(**inputs, max_new_tokens=5000)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


Given the following Common Requirement Modeling Language mode, generate a CRML model of a air conditioning unit.

model Interaction_ECS_Consumer is {
    /* Abstract class for a system */
    partial class System is {
        /* Boolean that states whether the system is in operation.
        The values of the Boolean can be provided by a behavioral model of the system via the FMI, or by the system itself via a sensor */
        Boolean inOperation is external;
    };

    /* Class that defines the ECS for the purpose of the contract between the ECS and the consumer */
    class ECS is {
        /* The time dependent values for deltaT from the ECS.
        They are provided by a behavioral model of the ECS via the FMI, or by the ECS itself via sensors. */
        TemperatureDifference deltaT is external;
    } extends System;

    /* Class that defines the consumer for the purpose of the contract between the ECS and the consumer */
    class Consumer is {
        /* The time dependent 